In [2]:
import pandas as pd
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_columns', 100)


In [21]:
input_file = "raw/ContractNLI_reuben/train-00000-of-00001-d5675d98fde8c367.parquet"
df_short = pd.read_parquet(input_file)

In [10]:
df_short.iloc[1]['evidence_texts'].tolist()

['5. All Confidential Information in any form and any medium, including all copies thereof, disclosed to the Recipient shall be returned to UNHCR or destroyed: ',
 '(a) if a business relationship is not entered into with UNHCR on or before the date which is three (3) months after the date both Parties have signed the Agreement; or ']

In [17]:
l1 = df_short.iloc[1]['evidence_texts'].tolist()
". ".join(l1)

'5. All Confidential Information in any form and any medium, including all copies thereof, disclosed to the Recipient shall be returned to UNHCR or destroyed: . (a) if a business relationship is not entered into with UNHCR on or before the date which is three (3) months after the date both Parties have signed the Agreement; or '

In [20]:
df = df_short

In [22]:
df['label'] = df['label'].replace({'notmentioned': 'not_mentioned'})
full_document = df.groupby(['document_id']).nth(0)[['document_id', 'text']]

In [39]:
df_short_doc_34 = df_short[df_short['document_id'] == 34]
df_short_doc_34['evidence_texts'].tolist()

[array([], dtype=object),
 array(['5. All Confidential Information in any form and any medium, including all copies thereof, disclosed to the Recipient shall be returned to UNHCR or destroyed: ',
        '(a) if a business relationship is not entered into with UNHCR on or before the date which is three (3) months after the date both Parties have signed the Agreement; or '],
       dtype=object),
 array(['4. Nothing in this Agreement is to be construed as granting the Recipient, by implication or otherwise, any right whatsoever with respect to the Confidential Information or part thereof.'],
       dtype=object),
 array(['11. The Recipient shall not advertise or otherwise make public the fact that it has a confidential relationship with UNHCR, nor shall the Recipient, in any manner whatsoever use the name, emblem, or official seal of the United Nations or UNHCR, or any abbreviation of the name of the United Nations or UNHCR in connection with its business or otherwise.'],
       dtype=o

In [59]:
def hypo_inferred(row):
    """
    Helper function
    """
    # Turn evidence into a single string separated by newlines
    evidence_list = row['evidence_texts'].tolist()
    evidence_list_str = ". ".join(evidence_list)

    hypothesis = row['hypothesis']
    hypo_label = row['label']
    hypo_id = row['hypothesis_id']

    assert None not in [hypo_id, hypothesis, evidence_list_str, hypo_label], f"One of the values in {[hypo_id, hypothesis, evidence_list_str, hypo_label]} is None"
    
    assert all(isinstance(x, str) for x in [hypo_id, hypothesis, evidence_list_str, hypo_label]), f"One of the values in {[hypo_id, hypothesis, evidence_list_str, hypo_label]} is not a string"

    data_dict = {
        'hypothesis_id': hypo_id,
        'hypothesis_label': hypo_label,
        'hypothesis': hypothesis,
        'source_clause': evidence_list_str
    }
    # return the dictionary as a string
    return data_dict

hypotheses_inferred = df.copy()[['document_id','evidence_texts','hypothesis','label','hypothesis_id']]
hypotheses_inferred['inference'] = hypotheses_inferred.apply(hypo_inferred, axis=1)
hypotheses_inferred = hypotheses_inferred[['document_id', 'inference']]

# aggregate the hypo_infer strings into a list by document_id
hypotheses_inferred_byid = hypotheses_inferred.groupby('document_id')['inference'].agg(list).reset_index()

In [60]:
hypotheses_inferred_byid

,document_id,inference
0,34,"[{'hypothesis_id': 'nda-11', 'hypothesis_label': 'not_mentioned', 'hypothesis': 'Receiving Party..."
1,86,"[{'hypothesis_id': 'nda-11', 'hypothesis_label': 'not_mentioned', 'hypothesis': 'Receiving Party..."
2,87,"[{'hypothesis_id': 'nda-11', 'hypothesis_label': 'not_mentioned', 'hypothesis': 'Receiving Party..."
3,88,"[{'hypothesis_id': 'nda-11', 'hypothesis_label': 'not_mentioned', 'hypothesis': 'Receiving Party..."
4,89,"[{'hypothesis_id': 'nda-11', 'hypothesis_label': 'not_mentioned', 'hypothesis': 'Receiving Party..."
...,...,...
418,620,"[{'hypothesis_id': 'nda-11', 'hypothesis_label': 'not_mentioned', 'hypothesis': 'Receiving Party..."
419,621,"[{'hypothesis_id': 'nda-11', 'hypothesis_label': 'not_mentioned', 'hypothesis': 'Receiving Party..."
420,622,"[{'hypothesis_id': 'nda-11', 'hypothesis_label': 'not_mentioned', 'hypothesis': 'Receiving Party..."
421,623,"[{'hypothesis_id': 'nda-11', 'hypothesis_label': 'not_mentioned', 'hypothesis': 'Receiving Party..."


In [56]:
hypotheses_inferred_byid.iloc[0]['inference']

[{'hypothesis_id': 'nda-11',
  'hypothesis': "Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information.",
  'source_clause': '',
  'hypothesis_label': 'not_mentioned'},
 {'hypothesis_id': 'nda-16',
  'hypothesis': 'Receiving Party shall destroy or return some Confidential Information upon the termination of Agreement.',
  'source_clause': '5. All Confidential Information in any form and any medium, including all copies thereof, disclosed to the Recipient shall be returned to UNHCR or destroyed: . (a) if a business relationship is not entered into with UNHCR on or before the date which is three (3) months after the date both Parties have signed the Agreement; or ',
  'hypothesis_label': 'entailment'},
 {'hypothesis_id': 'nda-15',
  'hypothesis': 'Agreement shall not grant Receiving Party any right to Confidential Information.',
  'source_clause': '4. Nothing in this Agreement is to be construed as granting the Recipient, by implicat

In [57]:
hypotheses_inferred_byid.dtypes

document_id     int64
inference      object
dtype: object